Knowledge Graph Creation from Triplet and Metadata

In [9]:
import os
import pandas as pd
import sys
from neo4j import GraphDatabase

# --- Configuration ---
URI = "URI"
AUTH = ("neo4j", "Password") 

# --- Filenames ---
from pathlib import Path

ROOT = Path.cwd().parent   # move one level up from CODE

INPUT = ROOT / "INPUT"

TRIPLET_FILE = INPUT / "master_triplets_Final.csv"
GENE_META_FILE = INPUT / "gene_metadata_Final.csv"
COMPOUND_META_FILE = INPUT / "phytochemical_metadata_Final.csv"
HERB_META_FILE = INPUT / "shd_metadata_Final.csv"

class KGBuilder:
    def __init__(self, uri, auth):
        self.driver = GraphDatabase.driver(uri, auth=auth)

    def close(self):
        self.driver.close()

    def create_constraints(self):
        print("[INFO] Creating constraints...")
        queries = [
            "CREATE CONSTRAINT IF NOT EXISTS FOR (n:SHD) REQUIRE n.id IS UNIQUE",
            "CREATE CONSTRAINT IF NOT EXISTS FOR (n:Phytochemical) REQUIRE n.id IS UNIQUE",
            "CREATE CONSTRAINT IF NOT EXISTS FOR (n:Target) REQUIRE n.id IS UNIQUE",
            "CREATE CONSTRAINT IF NOT EXISTS FOR (n:`FDA approved drug`) REQUIRE n.id IS UNIQUE", 
            "CREATE CONSTRAINT IF NOT EXISTS FOR (n:Disease) REQUIRE n.id IS UNIQUE"
        ]
        with self.driver.session() as session:
            for q in queries:
                session.run(q)

    def load_triplets(self):
        print(f"[INFO] Loading Triplets from {TRIPLET_FILE}...")
        if not os.path.exists(TRIPLET_FILE):
            print(f"[ERROR] File {TRIPLET_FILE} not found.")
            return

        df = pd.read_csv(TRIPLET_FILE)
        
        # --- DATA CLEANING FIX ---
        # 1. Convert to string to handle types safely
        df['source_id'] = df['source_id'].astype(str).str.strip()
        df['target_id'] = df['target_id'].astype(str).str.strip()
        df['properties'] = df['properties'].fillna("")

        # 2. Assign "NA" to missing IDs instead of dropping them
        #    This covers 'nan', 'NA', 'NaN', and empty strings
        na_values = ['nan', 'NA', 'NaN', '']
        df.loc[df['source_id'].isin(na_values), 'source_id'] = "NA"
        df.loc[df['target_id'].isin(na_values), 'target_id'] = "NA"

        # 3. Final verification
        na_count = len(df[df['source_id'] == 'NA']) + len(df[df['target_id'] == 'NA'])
        if na_count > 0:
            print(f"[WARNING] {na_count} entries have ID='NA'. Ensure this doesn't violate unique constraints.")

        grouped = df.groupby('relation')
        
        with self.driver.session() as session:
            for relation, group_df in grouped:
                batch_data = group_df.to_dict('records')
                print(f"       Processing relation: '{relation}' ({len(batch_data)} rows)...")
                
                # ---------------------------------------------------------
                # 1. SHD -> Phytochemical (contains_chemical)
                # ---------------------------------------------------------
                if relation == 'contains_chemical':
                    q = """
                    UNWIND $batch as row
                    MERGE (s:SHD {id: row.source_id}) 
                        ON CREATE SET s.name = row.source_id
                    MERGE (t:Phytochemical {id: row.target_id}) 
                        ON CREATE SET t.name = row.target_id 
                    
                    WITH s, t, row,
                         CASE 
                            WHEN row.properties CONTAINS 'part:' THEN 
                                 trim(split(row.properties, 'part:')[1])
                            ELSE 'Whole Plant' 
                         END AS part_val
                    
                    MERGE (s)-[r:contains_chemical]->(t)
                    SET r.part = part_val
                    """
                    session.run(q, batch=batch_data)
                
                # ---------------------------------------------------------
                # 2. Phytochemical -> Target (interacts_with)
                # ---------------------------------------------------------
                elif relation == 'interacts_with':
                    q = """
                    UNWIND $batch as row
                    MERGE (s:Phytochemical {id: row.source_id}) ON CREATE SET s.name = row.source_id
                    MERGE (t:Target {id: row.target_id}) ON CREATE SET t.name = row.target_id
                    MERGE (s)-[:interacts_with]->(t)
                    """
                    session.run(q, batch=batch_data)

                # ---------------------------------------------------------
                # 3. FDA approved drug -> Target (affects)
                # ---------------------------------------------------------
                elif relation == 'affects':
                    q = """
                    UNWIND $batch as row
                    MERGE (s:`FDA approved drug` {id: row.source_id}) ON CREATE SET s.name = row.source_id
                    MERGE (t:Target {id: row.target_id}) ON CREATE SET t.name = row.target_id
                    MERGE (s)-[:affects]->(t)
                    """
                    session.run(q, batch=batch_data)

                # ---------------------------------------------------------
                # 4. FDA approved drug -> Disease (treats)
                # ---------------------------------------------------------
                elif relation == 'treats':
                    q = """
                    UNWIND $batch as row
                    MERGE (s:`FDA approved drug` {id: row.source_id}) ON CREATE SET s.name = row.source_id
                    MERGE (t:Disease {id: row.target_id}) ON CREATE SET t.name = row.target_id
                    MERGE (s)-[:treats]->(t)
                    """
                    session.run(q, batch=batch_data)

                # ---------------------------------------------------------
                # 5. Target -> Disease (corresponds_to)
                # ---------------------------------------------------------
                elif relation == 'corresponds_to':
                    q = """
                    UNWIND $batch as row
                    MERGE (s:Target {id: row.source_id}) ON CREATE SET s.name = row.source_id
                    MERGE (t:Disease {id: row.target_id}) ON CREATE SET t.name = row.target_id
                    MERGE (s)-[:corresponds_to]->(t)
                    """
                    session.run(q, batch=batch_data)

    def load_metadata(self):
        print("[INFO] Loading Metadata...")
        
        # 1. SHD METADATA (Sanskrit Names)
        if os.path.exists(HERB_META_FILE):
            print(f"       Enriching SHDs from {HERB_META_FILE}...")
            df_herb = pd.read_csv(HERB_META_FILE, dtype=str)
            herb_data = df_herb.to_dict('records')
            
            # Updated label: SHD
            query = """
            UNWIND $batch as row
            MATCH (h:SHD {id: row.plant_name})
            SET h.sanskrit_name = row.sanskrit_name
            """
            with self.driver.session() as session:
                session.run(query, batch=herb_data)
            print(f"       [DONE] Updated SHDs with Sanskrit names.")
        else:
            print(f"       [WARNING] {HERB_META_FILE} not found.")

        # 2. TARGET METADATA
        if os.path.exists(GENE_META_FILE):
            print(f"       Enriching Targets from {GENE_META_FILE}...")
            df_gene = pd.read_csv(GENE_META_FILE, dtype=str)
            gene_data = df_gene.to_dict('records')
            
            query = """
            UNWIND $batch as row
            MATCH (t:Target) WHERE toString(t.id) = row.entrez_id
            SET t.name = row.symbol
            """
            with self.driver.session() as session:
                session.run(query, batch=gene_data)
            print(f"       [DONE] Updated Target names.")

        # 3. PHYTOCHEMICAL METADATA
        if os.path.exists(COMPOUND_META_FILE):
            print(f"       Enriching Phytochemicals from {COMPOUND_META_FILE}...")
            df_comp = pd.read_csv(COMPOUND_META_FILE, dtype=str)
            comp_data = df_comp.to_dict('records')
            
            query = """
            UNWIND $batch as row
            MATCH (c:Phytochemical) WHERE toString(c.id) = row.cid
            SET c.name = row.chemical_name
            """
            with self.driver.session() as session:
                session.run(query, batch=comp_data)
            print(f"       [DONE] Updated Phytochemical names.")

# --- Execution ---
if __name__ == "__main__":
    kg = KGBuilder(URI, AUTH)
    try:
        # Note: To start fresh, you may uncomment the line below:
        kg.driver.session().run("MATCH (n) DETACH DELETE n")
        
        kg.create_constraints()
        kg.load_triplets()
        kg.load_metadata()
        print("\n[SUCCESS] Knowledge Graph built successfully.")
    except Exception as e:
        print(f"[ERROR] An unexpected error occurred: {e}")
    finally:
        kg.close()

[INFO] Creating constraints...
[INFO] Loading Triplets from /home/rtiwari/Desktop/Nahi_analysis/Knowledge_Graphs/INPUT/master_triplets_Final.csv...
       Processing relation: 'affects' (103 rows)...
       Processing relation: 'contains_chemical' (224 rows)...
       Processing relation: 'corresponds_to' (532 rows)...
       Processing relation: 'interacts_with' (3082 rows)...
       Processing relation: 'treats' (32 rows)...
[INFO] Loading Metadata...
       Enriching SHDs from /home/rtiwari/Desktop/Nahi_analysis/Knowledge_Graphs/INPUT/shd_metadata_Final.csv...
       [DONE] Updated SHDs with Sanskrit names.
       Enriching Targets from /home/rtiwari/Desktop/Nahi_analysis/Knowledge_Graphs/INPUT/gene_metadata_Final.csv...
       [DONE] Updated Target names.
       Enriching Phytochemicals from /home/rtiwari/Desktop/Nahi_analysis/Knowledge_Graphs/INPUT/phytochemical_metadata_Final.csv...
       [DONE] Updated Phytochemical names.

[SUCCESS] Knowledge Graph built successfully.


Knowledge Graph Stats

In [10]:
import pandas as pd
from neo4j import GraphDatabase
import os

# --- CONFIGURATION ---
URI = "URI"
AUTH = ("neo4j", "Password")
OUTPUT_FILE = ".../OUTPUT/Graph_Statistics_Report.csv"

class GraphStats:
    def __init__(self, uri, auth):
        self.driver = GraphDatabase.driver(uri, auth=auth)

    def close(self):
        self.driver.close()

    def generate_report(self):
        print("[INFO] Fetching graph statistics...")
        stats_data = []

        with self.driver.session() as session:
            # --- 1. NODE COUNTS ---
            # Finds every label (SHD, Target, etc.) and counts the nodes
            node_query = """
            MATCH (n)
            RETURN distinct labels(n)[0] AS Entity, count(n) AS Count, 'Node' AS Type
            ORDER BY Count DESC
            """
            result_nodes = session.run(node_query)
            for record in result_nodes:
                stats_data.append({
                    "Category": record["Entity"],
                    "Count": record["Count"],
                    "Type": "Node"
                })

            # --- 2. RELATIONSHIP COUNTS ---
            # Finds every relationship type (interacts_with, etc.) and counts them
            edge_query = """
            MATCH ()-[r]->()
            RETURN type(r) AS Relationship, count(r) AS Count, 'Edge' AS Type
            ORDER BY Count DESC
            """
            result_edges = session.run(edge_query)
            for record in result_edges:
                stats_data.append({
                    "Category": record["Relationship"],
                    "Count": record["Count"],
                    "Type": "Edge"
                })

            # --- 3. GRAND TOTALS ---
            total_nodes = session.run("MATCH (n) RETURN count(n) AS total").single()["total"]
            total_edges = session.run("MATCH ()-[r]->() RETURN count(r) AS total").single()["total"]
            
            stats_data.append({"Category": "TOTAL NODES", "Count": total_nodes, "Type": "Summary"})
            stats_data.append({"Category": "TOTAL EDGES", "Count": total_edges, "Type": "Summary"})

        # --- SAVE TO CSV ---
        if stats_data:
            df = pd.DataFrame(stats_data)
            
            # Reorder columns for readability
            df = df[['Type', 'Category', 'Count']]
            
            # Ensure directory exists
            os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)
            
            df.to_csv(OUTPUT_FILE, index=False)
            print(f"[SUCCESS] Statistics saved to: {OUTPUT_FILE}")
            print("\n--- Preview ---")
            print(df.to_string(index=False))
        else:
            print("[WARNING] The database appears to be empty.")

if __name__ == "__main__":
    app = GraphStats(URI, AUTH)
    try:
        app.generate_report()
    finally:
        app.close()
    

[INFO] Fetching graph statistics...
[SUCCESS] Statistics saved to: .../OUTPUT/Graph_Statistics_Report.csv

--- Preview ---
   Type          Category  Count
   Node            Target   1099
   Node     Phytochemical    188
   Node FDA approved drug     32
   Node               SHD     11
   Node           Disease      2
   Edge    interacts_with   3082
   Edge    corresponds_to    532
   Edge contains_chemical    224
   Edge           affects    103
   Edge            treats     32
Summary       TOTAL NODES   1332
Summary       TOTAL EDGES   3973


Intersection_Analysis

In [11]:
import pandas as pd
import os
from neo4j import GraphDatabase

# --- CONFIGURATION ---
URI = "URI"
AUTH = ("neo4j", "Password") 
MIN_HERBS = 3

# --- OUTPUT PATHS ---
from pathlib import Path
# ---------- Detect project root ----------
ROOT = Path.cwd()

while not (ROOT / "OUTPUT").exists():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Project root with OUTPUT folder not found")
    ROOT = ROOT.parent
# ---------- Build paths ----------
BASE_DIR = ROOT / "OUTPUT" / "OUTPUT_INTERSECTION"

FILE_DIABETES = BASE_DIR / "KG_Diabetes_Total_HighConfidence.csv"
FILE_OBESITY = BASE_DIR / "KG_Obesity_Total_HighConfidence.csv"
FILE_SHARED = BASE_DIR / "KG_Shared_Total_HighConfidence.csv"

class ConsensusAnalyzer:
    def __init__(self, uri, auth):
        self.driver = GraphDatabase.driver(uri, auth=auth)

    def close(self):
        self.driver.close()

    def run_query_and_save(self, label, query, filename):
        print(f"[INFO] Running analysis for: {label}...")
        
        # Ensure directory exists
        os.makedirs(os.path.dirname(filename), exist_ok=True)
        
        with self.driver.session() as session:
            result = session.run(query)
            data = [record.data() for record in result]
        
        if data:
            df = pd.DataFrame(data)
            df.to_csv(filename, index=False)
            print(f"       [SUCCESS] Saved {len(df)} targets to {filename}")
        else:
            print(f"       [WARNING] No targets found for {label} meeting the criteria.")

    def analyze(self):
        # 1. DIABETES (Total)
        # Updated Label: SHDs -> SHD
        q_diabetes = f"""
        MATCH (t:Target)-[:corresponds_to]->(d:Disease {{name: 'Type 2 Diabetes'}})
        MATCH (h:SHD)-[:contains_chemical]->(c:Phytochemical)-[:interacts_with]->(t)
        WITH t, count(DISTINCT h) as herb_count, collect(DISTINCT h.name) as herbs
        WHERE herb_count >= {MIN_HERBS}
        RETURN t.id as Entrez_ID, t.name as Gene_Symbol, herb_count, herbs
        ORDER BY herb_count DESC
        """

        # 2. OBESITY (Total)
        # Updated Label: SHDs -> SHD
        q_obesity = f"""
        MATCH (t:Target)-[:corresponds_to]->(d:Disease {{name: 'Obesity'}})
        MATCH (h:SHD)-[:contains_chemical]->(c:Phytochemical)-[:interacts_with]->(t)
        WITH t, count(DISTINCT h) as herb_count, collect(DISTINCT h.name) as herbs
        WHERE herb_count >= {MIN_HERBS}
        RETURN t.id as Entrez_ID, t.name as Gene_Symbol, herb_count, herbs
        ORDER BY herb_count DESC
        """

        # 3. SHARED (Intersection)
        # Updated Label: SHDs -> SHD
        q_shared = f"""
        MATCH (t:Target)-[:corresponds_to]->(:Disease {{name: 'Type 2 Diabetes'}})
        MATCH (t)-[:corresponds_to]->(:Disease {{name: 'Obesity'}})
        MATCH (h:SHD)-[:contains_chemical]->(c:Phytochemical)-[:interacts_with]->(t)
        WITH t, count(DISTINCT h) as herb_count, collect(DISTINCT h.name) as herbs
        WHERE herb_count >= {MIN_HERBS}
        RETURN t.id as Entrez_ID, t.name as Gene_Symbol, herb_count, herbs
        ORDER BY herb_count DESC
        """

        # Execute
        self.run_query_and_save("Diabetes (Total)", q_diabetes, FILE_DIABETES)
        self.run_query_and_save("Obesity (Total)", q_obesity, FILE_OBESITY)
        self.run_query_and_save("Shared (Intersection)", q_shared, FILE_SHARED)

# --- EXECUTION ---
if __name__ == "__main__":
    analyzer = ConsensusAnalyzer(URI, AUTH)
    try:
        analyzer.analyze()
        print("\n[DONE] Analysis complete. Files generated successfully.")
    except Exception as e:
        print(f"[ERROR] An error occurred: {e}")
    finally:
        analyzer.close()

[INFO] Running analysis for: Diabetes (Total)...
       [SUCCESS] Saved 22 targets to /home/rtiwari/Desktop/Nahi_analysis/Knowledge_Graphs/OUTPUT/OUTPUT_INTERSECTION/KG_Diabetes_Total_HighConfidence.csv
[INFO] Running analysis for: Obesity (Total)...
       [SUCCESS] Saved 34 targets to /home/rtiwari/Desktop/Nahi_analysis/Knowledge_Graphs/OUTPUT/OUTPUT_INTERSECTION/KG_Obesity_Total_HighConfidence.csv
[INFO] Running analysis for: Shared (Intersection)...
       [SUCCESS] Saved 7 targets to /home/rtiwari/Desktop/Nahi_analysis/Knowledge_Graphs/OUTPUT/OUTPUT_INTERSECTION/KG_Shared_Total_HighConfidence.csv

[DONE] Analysis complete. Files generated successfully.


In [12]:
import pandas as pd
from neo4j import GraphDatabase
import os
import sys

# --- CONFIGURATION ---
from pathlib import Path
# ---------- Neo4j config ----------
URI = "URI"
AUTH = ("neo4j", "Password")
# ---------- Detect project root ----------
ROOT = Path.cwd()
while not (ROOT / "OUTPUT").exists():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Project root with OUTPUT folder not found")
    ROOT = ROOT.parent
# ---------- Output directory ----------
BASE_OUTPUT_DIR = ROOT / "OUTPUT" / "OUTPUT_INTERSECTION"

BASE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

class HerbIntersectionAnalyzer:
    def __init__(self, uri, auth):
        self.driver = GraphDatabase.driver(uri, auth=auth)

    def close(self):
        self.driver.close()

    def create_directories(self):
        """Creates the required subfolders for the analysis."""
        self.dirs = {
            "Diabetes": os.path.join(BASE_OUTPUT_DIR, "1_Diabetes_Intersection"),
            "Obesity": os.path.join(BASE_OUTPUT_DIR, "2_Obesity_Intersection"),
            "Shared": os.path.join(BASE_OUTPUT_DIR, "3_Shared_Comorbidity_Intersection")
        }
        for path in self.dirs.values():
            os.makedirs(path, exist_ok=True)
        print(f"[INFO] Output directories created at: {BASE_OUTPUT_DIR}")

    def get_all_herbs(self):
        """Fetches a list of all distinct SHDs (Herbs) in the graph."""
        # Updated Label: SHDs -> SHD
        query = "MATCH (h:SHD) RETURN distinct h.name as herb_name ORDER BY h.name"
        with self.driver.session() as session:
            return [record["herb_name"] for record in session.run(query)]

    def analyze_single_herb(self, herb_name):
        safe_name = str(herb_name).replace(" ", "_").replace("/", "-")
        
        with self.driver.session() as session:
            # 1. DIABETES INTERSECTION (Targets of this SHD -> Diabetes)
            # Updated Label: SHD
            q_dia = """
            MATCH (h:SHD {name: $herb})-[:contains_chemical]->(c:Phytochemical)-[:interacts_with]->(t:Target)
            MATCH (t)-[:corresponds_to]->(d:Disease {name: 'Type 2 Diabetes'})
            RETURN DISTINCT t.id as Entrez_ID, t.name as Gene_Symbol, collect(DISTINCT c.name) as Phytochemicals
            """
            result_dia = session.run(q_dia, herb=herb_name).data()
            if result_dia:
                pd.DataFrame(result_dia).to_csv(f"{self.dirs['Diabetes']}/{safe_name}.csv", index=False)

            # 2. OBESITY INTERSECTION (Targets of this SHD -> Obesity)
            # Updated Label: SHD
            q_obe = """
            MATCH (h:SHD {name: $herb})-[:contains_chemical]->(c:Phytochemical)-[:interacts_with]->(t:Target)
            MATCH (t)-[:corresponds_to]->(d:Disease {name: 'Obesity'})
            RETURN DISTINCT t.id as Entrez_ID, t.name as Gene_Symbol, collect(DISTINCT c.name) as Phytochemicals
            """
            result_obe = session.run(q_obe, herb=herb_name).data()
            if result_obe:
                pd.DataFrame(result_obe).to_csv(f"{self.dirs['Obesity']}/{safe_name}.csv", index=False)

            # 3. SHARED INTERSECTION (Targets of this SHD -> BOTH Diabetes AND Obesity)
            # Updated Label: SHD
            q_shared = """
            MATCH (h:SHD {name: $herb})-[:contains_chemical]->(c:Phytochemical)-[:interacts_with]->(t:Target)
            MATCH (t)-[:corresponds_to]->(d1:Disease {name: 'Type 2 Diabetes'})
            MATCH (t)-[:corresponds_to]->(d2:Disease {name: 'Obesity'})
            RETURN DISTINCT t.id as Entrez_ID, t.name as Gene_Symbol, collect(DISTINCT c.name) as Phytochemicals
            """
            result_shared = session.run(q_shared, herb=herb_name).data()
            if result_shared:
                pd.DataFrame(result_shared).to_csv(f"{self.dirs['Shared']}/{safe_name}.csv", index=False)

    def run_full_analysis(self):
        self.create_directories()
        
        print("[INFO] Fetching SHD (Herb) list...")
        herbs = self.get_all_herbs()
        total_herbs = len(herbs)
        print(f"[INFO] Found {total_herbs} SHDs. Starting intersection analysis...")

        for i, herb in enumerate(herbs):
            # Dynamic progress printing
            sys.stdout.write(f"\r[PROCESSING] {i+1}/{total_herbs}: {herb:<30}")
            sys.stdout.flush()
            self.analyze_single_herb(herb)
        
        print("\n[SUCCESS] Analysis complete. All intersection files generated.")

# --- EXECUTION ---
if __name__ == "__main__":
    analyzer = HerbIntersectionAnalyzer(URI, AUTH)
    try:
        analyzer.run_full_analysis()
    except Exception as e:
        print(f"\n[ERROR] An unexpected error occurred: {e}")
    finally:
        analyzer.close()

[INFO] Output directories created at: /home/rtiwari/Desktop/Nahi_analysis/Knowledge_Graphs/OUTPUT/OUTPUT_INTERSECTION
[INFO] Fetching SHD (Herb) list...
[INFO] Found 11 SHDs. Starting intersection analysis...
[PROCESSING] 11/11: Terminalia arjuna             
[SUCCESS] Analysis complete. All intersection files generated.


Data for Sankey and Mechanism of action

In [16]:
import pandas as pd
from neo4j import GraphDatabase
import os

# --- CONFIGURATION ---
from pathlib import Path
# ---------- Neo4j config ----------
URI = "URI"
AUTH = ("neo4j", "Password")
# ---------- Detect project root ----------
ROOT = Path.cwd()
while not (ROOT / "OUTPUT").exists():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Project root with OUTPUT folder not found")
    ROOT = ROOT.parent
# ---------- Output directory ----------
OUTPUT_DIR = (
    ROOT
    / "OUTPUT"
    / "Polypharmacology_New"
    / "Sankey_Plot_for_MOA"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Sankey INPUT dir:", OUTPUT_DIR)

class PolypharmacologyAnalyzer:
    def __init__(self, uri, auth):
        self.driver = GraphDatabase.driver(uri, auth=auth)

    def close(self):
        self.driver.close()

    def run_consolidated_analysis(self):
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        print("[INFO] Analyzing Combined Polypharmacology (Diabetes & Obesity)...")

        # Updated Query: Aliases renamed to match KG Node Labels
        master_query = """
        MATCH (h:SHD)-[:contains_chemical]->(c:Phytochemical)
        MATCH (c)-[:interacts_with]->(t:Target)
        MATCH (d:`FDA approved drug`)-[:affects]->(t)
        MATCH (t)-[:corresponds_to]->(dis:Disease)
        MATCH (d)-[:treats]->(dis)

        WHERE h.name <> d.name
          AND (toLower(dis.name) CONTAINS 'diabet' OR toLower(dis.name) CONTAINS 'obes')
        
        RETURN 
            h.name as SHD,
            c.name as Phytochemical,
            t.name as Target,
            d.name as `FDA approved drug`,
            dis.name as Disease,
            CASE 
                WHEN toLower(dis.name) CONTAINS 'diabet' AND toLower(dis.name) CONTAINS 'obes' THEN 'Both'
                WHEN toLower(dis.name) CONTAINS 'diabet' THEN 'Diabetes'
                WHEN toLower(dis.name) CONTAINS 'obes' THEN 'Obesity'
                ELSE 'Other'
            END as Indication_Type
        """
        
        with self.driver.session() as session:
            result = session.run(master_query)
            data = [record.data() for record in result]

        if not data:
            print("[WARNING] No overlapping data found. Check graph connections.")
            return

        df = pd.DataFrame(data)

        # 1. Save the Master Sheet
        master_file = os.path.join(OUTPUT_DIR, "Master_Herb_Drug_Mimicry_Sheet.csv")
        df.to_csv(master_file, index=False)
        print(f"[SAVED] Master Sheet (with KG Labels): {master_file}")

        # 2. Save a Relationship Summary
        # Grouping keys updated to match new column names
        summary_df = df.groupby(['SHD', 'FDA approved drug', 'Indication_Type']).agg({
            'Phytochemical': lambda x: ", ".join(set(x)),
            'Target': lambda x: ", ".join(set(x)),
            'Disease': lambda x: ", ".join(set(x))
        }).reset_index()

        summary_file = os.path.join(OUTPUT_DIR, "Herb_Drug_Synergy_Summary.csv")
        summary_df.to_csv(summary_file, index=False)
        print(f"[SAVED] Synergy Summary: {summary_file}")

if __name__ == "__main__":
    analyzer = PolypharmacologyAnalyzer(URI, AUTH)
    try:
        analyzer.run_consolidated_analysis()
        print("\n[SUCCESS] Analysis Complete!")
    except Exception as e:
        print(f"\n[ERROR] An error occurred: {e}")
    finally:
        analyzer.close()

Sankey INPUT dir: /home/rtiwari/Desktop/Nahi_analysis/Knowledge_Graphs/OUTPUT/Polypharmacology_New/Sankey_Plot_for_MOA
[INFO] Analyzing Combined Polypharmacology (Diabetes & Obesity)...
[SAVED] Master Sheet (with KG Labels): /home/rtiwari/Desktop/Nahi_analysis/Knowledge_Graphs/OUTPUT/Polypharmacology_New/Sankey_Plot_for_MOA/Master_Herb_Drug_Mimicry_Sheet.csv
[SAVED] Synergy Summary: /home/rtiwari/Desktop/Nahi_analysis/Knowledge_Graphs/OUTPUT/Polypharmacology_New/Sankey_Plot_for_MOA/Herb_Drug_Synergy_Summary.csv

[SUCCESS] Analysis Complete!


Phytochemical Based Interscetion

In [14]:
import pandas as pd
from neo4j import GraphDatabase
import os

# --- CONFIGURATION ---
from pathlib import Path
# ---------- Neo4j config ----------
URI = "URI"
AUTH = ("neo4j", "Password")
# ---------- Detect project root ----------
ROOT = Path.cwd()
while not (ROOT / "OUTPUT").exists():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Project root with OUTPUT folder not found")
    ROOT = ROOT.parent
# ---------- Output file ----------
OUTPUT_FILE = ROOT / "OUTPUT" / "Phytochemical_Distribution_Report.csv"
print("Writing file to:", OUTPUT_FILE)

class PhytochemicalReport:
    def __init__(self, uri, auth):
        self.driver = GraphDatabase.driver(uri, auth=auth)

    def close(self):
        self.driver.close()

    def generate_csv(self):
        print("[INFO] Querying database...")
        
        # Updated Label: SHD (Singular)
        query = """
        MATCH (h:SHD)-[:contains_chemical]->(p:Phytochemical)
        RETURN 
            p.name AS chemical_name,
            p.id AS external_id,
            count(DISTINCT h) AS shd_count,
            collect(DISTINCT h.name) AS plant_list
        ORDER BY shd_count DESC
        """

        with self.driver.session() as session:
            result = session.run(query)
            data = [record.data() for record in result]

        if not data:
            print("[WARNING] No data found.")
            return

        # Create DataFrame
        df = pd.DataFrame(data)

        # 1. Format the 'Plant name' column to use pipes "|"
        df['Plant name'] = df['plant_list'].apply(lambda x: " | ".join(sorted(x)))

        # 2. Logic to set "NA" if ID is missing or same as name (fallback case)
        def fix_id(row):
            cid = str(row['external_id']).strip()
            name = str(row['chemical_name']).strip()
            
            # If ID is missing, 'nan', or exactly matches the name -> return "NA"
            if not cid or cid.lower() in ['nan', 'na', ''] or cid == name:
                return "NA"
            return cid

        df['external_id'] = df.apply(fix_id, axis=1)

        # 3. Rename columns
        df = df.rename(columns={
            'chemical_name': 'Chemical name',
            'external_id': 'External chemical identifier',
            'shd_count': 'Number of SHDs containing the phytochemical'
        })

        # 4. Select specific columns
        df = df[[
            'Chemical name', 
            'External chemical identifier', 
            'Number of SHDs containing the phytochemical', 
            'Plant name'
        ]]

        # Ensure output directory exists
        os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

        # 5. Save to CSV
        df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8')
        
        print(f"[SUCCESS] CSV saved to: {OUTPUT_FILE}")
        print("\n--- Preview of Generated CSV ---")
        print(df.head().to_string(index=False))

if __name__ == "__main__":
    app = PhytochemicalReport(URI, AUTH)
    try:
        app.generate_csv()
    finally:
        app.close()

Writing file to: /home/rtiwari/Desktop/Nahi_analysis/Knowledge_Graphs/OUTPUT/Phytochemical_Distribution_Report.csv
[INFO] Querying database...
[SUCCESS] CSV saved to: /home/rtiwari/Desktop/Nahi_analysis/Knowledge_Graphs/OUTPUT/Phytochemical_Distribution_Report.csv

--- Preview of Generated CSV ---
            Chemical name External chemical identifier  Number of SHDs containing the phytochemical                                                                                                                               Plant name
          Beta-sitosterol                   CID_222284                                            7 Aegle marmelos | Butea monosperma | Commiphora wightii | Diospyros malabarica | Tecomella undulata | Tectona grandis | Terminalia arjuna
                   Lupeol                   CID_259846                                            4                                                        Aegle marmelos | Diospyros malabarica | Gymnema sylvestre | Pterocarpus 